In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('features_v1_partial.csv')

In [2]:
# Calculate crop frequencies
crop_freq = df['crop'].value_counts(normalize=True) # normalize=True gives frequency, False gives count

# Map frequencies back to the crop column
df['crop_encoded'] = df['crop'].map(crop_freq)

# Drop the original crop column
df = df.drop(columns=['crop'])

print("--- Crop Frequency Encoding ---")
print(df[['crop_encoded']].head())
print(f"\nUnique frequency values created: {df['crop_encoded'].nunique()}")

--- Crop Frequency Encoding ---
   crop_encoded
0      0.015601
1      0.015601
2      0.015601
3      0.015601
4      0.015601

Unique frequency values created: 39


In [3]:
# Reload the original soil nutrient and type dataframes (corrected) for mapping
# (Copy this corrected data structure from the Task 2.1 script)
soil_type_data = {
    'district': [
        'anugul', 'baleshwar', 'bargarh', 'bhadrak', 'balangir', 'boudh', 
        'cuttack', 'deogarh', 'dhenkanal', 'gajapati', 'ganjam', 
        'jagatsinghapur', 'jajapur', 'jharsuguda', 'kalahandi', 'kandhamal', 
        'kendrapara', 'kendujhar', 'khordha', 'koraput', 'malkangiri', 
        'mayurbhanj', 'nabarangpur', 'nayagarh', 'nuapada', 'puri', 
        'rayagada', 'sambalpur', 'sonepur', 'sundargarh'
    ],
    'primary_soil_type': [
        'Black Soil', 'Coastal Salt Affected Alluvial, Deltaic Alluvial', # baleshwar
        'Mixed Red & Yellow, Black, Mixed Red & Black', 
        'Coastal Salt Affected Alluvial, Deltaic Alluvial',
        'Red, Black, Mixed Red & Black', 'Black Soil', 
        'Laterite, Deltaic Alluvial', 'Mixed Red & Yellow',
        'Red, Laterite', 'Deltaic Alluvial',
        'Red, Coastal Salt Affected Alluvial, Deltaic Alluvial, Black, Brown Forest',
        'Coastal Salt Affected Alluvial, Deltaic Alluvial', 'Deltaic Alluvial', # jajapur
        'Unknown', # Imputed for Jharsuguda
        'Red, Black', 'Brown Forest', 
        'Coastal Salt Affected Alluvial, Deltaic Alluvial',
        'Red, Laterite', 'Laterite, Coastal Salt Affected Alluvial', 'Red', # kendujhar
        'Red, Black', 'Red, Laterite', 'Red', 'Laterite, Brown Forest',
        'Red, Black', 'Laterite, Coastal Salt Affected Alluvial, Deltaic Alluvial, Black',
        'Red, Brown Forest', 'Laterite, Red & Yellow, Black, Mixed Red & Black',
        'Black, Mixed Red & Black', 'Mixed Red & Yellow'
    ]
}
soil_type_df_orig = pd.DataFrame(soil_type_data)

nutrient_data = [
    # (Using corrected district spellings)
    ('baleshwar', 6.0, 139.235, 14.115, 160.855),
    ('cuttack', 5.451429, 166.59, 11.271429, 218.723571),
    ('ganjam', 6.5, 119.15, 101.45, 245.1),
    ('khordha', 5.38, np.nan, 12.01, np.nan),
    ('puri', 5.51, 216.090909, 20.724545, 233.181818)
]
soil_nutrient_agg_df_orig = pd.DataFrame(nutrient_data, columns=['district', 'mean_ph', 'mean_n', 'mean_p', 'mean_k'])

# Merge nutrient data with soil types (only for districts where we have nutrient data)
soil_mapping_df = pd.merge(soil_nutrient_agg_df_orig, soil_type_df_orig, on='district', how='inner')

# Calculate mean nutrients per PRIMARY soil type (approximation for combined types)
# Note: This is a simplification. More complex methods could handle combinations better.
# We take the first listed soil type as the primary for calculating means.
soil_mapping_df['first_soil_type'] = soil_mapping_df['primary_soil_type'].apply(lambda x: x.split(',')[0].strip())
soil_type_means = soil_mapping_df.groupby('first_soil_type')[['mean_ph', 'mean_n', 'mean_p', 'mean_k']].mean()

print("\n--- Mean Nutrients Calculated per Primary Soil Type ---")
print(soil_type_means)

# Impute missing values in the main DataFrame
nutrient_cols = ['mean_ph', 'mean_n', 'mean_p', 'mean_k']

# Get the list of soil type column names we created in Task 4
soil_binary_cols = df.filter(like='soil_').columns

for col in nutrient_cols:
    # Create a mapping dictionary: soil_type -> mean_value
    impute_map = soil_type_means[col].to_dict()

    # Apply imputation row-wise
    # For each row, find which soil type column is 1, get the soil type name, and use the map
    def get_impute_value(row):
        if pd.isna(row[col]): # Only impute if NaN
            for soil_col in soil_binary_cols:
                if row[soil_col] == 1:
                    # Extract soil type name from column name (e.g., 'soil_red' -> 'Red')
                    soil_type_name = soil_col.split('_', 1)[1].replace('_', ' ').replace(' and ', ' & ').title() 
                    # Use the first part if it was combined (e.g., Coastal Salt Affected Alluvial)
                    soil_type_key = soil_type_name.split(',')[0].strip() 

                    # Handle potential mismatches (like Black vs Black Soil) - adjust keys as needed
                    if soil_type_key == "Black Soil": soil_type_key = "Black Soil" 
                    if soil_type_key == "Coastal Salt Affected Alluvial": soil_type_key = "Coastal Salt Affected Alluvial"

                    # Return the mapped mean if found, otherwise keep NaN (or use a global mean)
                    return impute_map.get(soil_type_key, np.nan) 
        return row[col] # Return original value if not NaN

    df[col] = df.apply(get_impute_value, axis=1)

    # Final fallback: Impute any remaining NaNs (e.g., 'Unknown' soil or types not in means) with the global mean
    global_mean = df[col].mean()
    df[col] = df[col].fillna(global_mean)

print("\n--- Null Check After Imputation ---")
print(df[nutrient_cols].isnull().sum())


--- Mean Nutrients Calculated per Primary Soil Type ---
                                 mean_ph      mean_n      mean_p      mean_k
first_soil_type                                                             
Coastal Salt Affected Alluvial  6.000000  139.235000   14.115000  160.855000
Laterite                        5.447143  191.340455   14.668658  225.952695
Red                             6.500000  119.150000  101.450000  245.100000

--- Null Check After Imputation ---
mean_ph    0
mean_n     0
mean_p     0
mean_k     0
dtype: int64


In [4]:
print("\n--- Final DataFrame Info ---")
df.info()

print("\n--- Final DataFrame Head ---")
print(df.head())

# Save the fully engineered feature set
df.to_csv('features_v2_final.csv', index=False)


--- Final DataFrame Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16153 entries, 0 to 16152
Data columns (total 57 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   year                                 16153 non-null  int64  
 1   area                                 16153 non-null  float64
 2   avg_temp_c                           16153 non-null  float64
 3   total_precip_mm                      16153 non-null  float64
 4   mean_ph                              16153 non-null  float64
 5   mean_n                               16153 non-null  float64
 6   mean_p                               16153 non-null  float64
 7   mean_k                               16153 non-null  float64
 8   yield_log1p                          16153 non-null  float64
 9   soil_black                           16153 non-null  int64  
 10  soil_black_soil                      16153 non-null  int64  
 11